# ME324 · Lab 2 — A neuron, by hand, in NumPy

**Lecture 2 · "Deep models — a primer" · 2026-08-04**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/REPLACE-WITH-COURSE-REPO/blob/main/labs/lab-02-perceptron-numpy.ipynb)
<!-- Instructor: replace REPLACE-WITH-COURSE-REPO with the course GitHub path,
     or students can use  File ▸ Upload notebook  in Colab. -->

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

**As ever, use an LLM if helpful.**
*"In NumPy, how do I …?"* is exactly the kind of question these tools are excellent at, and
looking things up this way is what every working researcher does. Two habits worth keeping:
ask for the **explanation** rather than just the line, and **run everything** it hands you.
The exam is closed-book, so what counts is that you can read the code back and say what it
does.

---

### What you'll do today (90 minutes)

By the end of this lab you will have:

1. Implemented a **perceptron** in base Python with NumPy, with explicit weights and a bias.
2. Run a **forward pass** — first on one voter by hand, then *vectorised* over a whole dataset — and turned scores into predictions with a **sigmoid**.
3. Built a **two-layer network** (hidden layer + ReLU + output) and seen that it is "just matmuls plus a non-linearity".
4. Compared trained baselines (**logistic regression** vs an **MLP**) and seen exactly *where the two-layer model beats the perceptron, and where it doesn't*.

> No PyTorch today. We keep everything in plain NumPy to consolidate the maths from the lecture.

## ⏱️ Plan for today (~90 minutes)

This lab is built for a single 90-minute session, and it's **completely fine not to finish every cell in the room**.

- **Core — do these:** the perceptron + vectorised forward pass + the 2-layer forward pass (Sections 1–3).
- **Stretch / take-home — skip if short on time:** the sklearn logistic-vs-MLP comparison and decision-boundary plots (Section 4) — skim, or finish at home.

_Most of the code is written for you; the `# TODO` cells are the parts you write. Worked answers are in the **Solutions** section at the bottom._

## Run me first

Run the cell below once to import the libraries and fix the random seed. In Colab everything you need is already installed.

In [ ]:
# === Run me first ===
# Lab 2 uses only numpy, matplotlib and scikit-learn — all pre-installed in Colab.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Reproducibility: fix NumPy's random seed so everyone sees the same numbers.
np.random.seed(0)

print("NumPy version:", np.__version__)
print("Setup complete - you're ready to go.")

## Section 0 · The data — synthetic voter turnout

We'll work with a small **synthetic voter-turnout dataset**. Each row is one (made-up) voter described by three features, all scaled so they're easy to compare:

| column | name | meaning |
|---|---|---|
| 0 | `age01` | age, scaled to [0, 1]  (0 ≈ 18, 1 ≈ 80) |
| 1 | `income01` | income, scaled to [0, 1]  (0 ≈ £15k, 1 ≈ £100k) |
| 2 | `voted2019` | did they vote in 2019?  (0 = no, 1 = yes) |

The label `y` is whether the voter **turned out** (1) or not (0).

These are *exactly* the features from the worked example in Lecture 2. We generate the data with a known rule that contains a deliberate **age × income interaction** — a non-linearity that, as we'll see, a single straight-line model cannot fully capture.

> **Continuity note.** This is the same `make_turnout_data` you'll meet again in **Lab 5**, where we rebuild today's models in PyTorch. Same data, so you can compare "by hand" vs "framework" directly.

First, the data generator (read it, then run it):

In [ ]:
def make_turnout_data(n=2000, seed=0):
    """Synthetic turnout data with a known nonlinear signal.
    Features: age01, income01, voted2019 (all in [0,1] / {0,1}).
    Label: turned out (0/1). Returns numpy arrays X (n,3), y (n,).
    The XOR-ish interaction between age and income makes a single linear
    model imperfect, so the 2-layer net can beat logistic regression."""
    import numpy as np
    rng = np.random.default_rng(seed)
    age01 = rng.uniform(0, 1, n)
    income01 = rng.uniform(0, 1, n)
    voted2019 = rng.integers(0, 2, n).astype(float)
    # logit with an interaction term (nonlinearity) + past-behaviour effect
    z = (1.6 * age01 + 1.2 * income01
         + 2.0 * voted2019
         - 3.0 * age01 * income01      # interaction: the nonlinearity
         - 1.0)
    p = 1 / (1 + np.exp(-z))
    y = (rng.uniform(0, 1, n) < p).astype(np.int64)
    X = np.stack([age01, income01, voted2019], axis=1).astype(np.float32)
    return X, y

Now generate the data and take a look:

In [ ]:
# Generate the dataset: 4000 synthetic voters.
X, y = make_turnout_data(n=4000, seed=0)

print("X shape:", X.shape, "  y shape:", y.shape)
print("Feature columns:  0 = age01   1 = income01   2 = voted2019")
print("\nFirst 5 voters (rows of X):")
print(X[:5])
print("\nTheir turnout labels y:", y[:5])
print(f"\nOverall turnout rate in the data: {y.mean():.1%}")

# A held-out test set, used in Section 4 to judge the trained models fairly.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)
print("Train rows:", X_train.shape[0], "  Test rows:", X_test.shape[0])

## Section 1 · One neuron, by hand

A **perceptron** (a single neuron) takes a vector of inputs $\mathbf{x}$, weights each one, sums them, and adds a bias:

$$
y = f(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b
= w_1 x_1 + w_2 x_2 + w_3 x_3 + b
$$

Pushing an input through the model left-to-right like this is called the **forward pass**. Every neural network — from this perceptron to GPT — works this way.

Let's reproduce the lecture's worked example exactly: a 45-year-old on ~£40k who voted in 2019, with the hand-picked weights from the slides.

In [ ]:
# The worked example from Lecture 2: a 45-year-old on ~£40k who voted in 2019.
x_demo = np.array([0.44, 0.29, 1.0])   # [age01, income01, voted2019]

# The weights and bias guessed by hand in the lecture.
w_demo = np.array([1.5, 0.7, 2.0])     # one weight per feature
b_demo = -1.5                          # the bias (baseline)

**Your turn (TODO).** Implement the forward pass. It's a single line: the dot product of `w` and `x`, plus `b`.

In [ ]:
def forward(x, w, b):
    """One neuron's forward pass: the weighted sum of inputs, plus the bias.

    Maths:  y = w . x + b  =  w[0]*x[0] + w[1]*x[1] + w[2]*x[2] + b

    Args:
        x : 1-D array of features for ONE voter
        w : 1-D array of weights (same length as x)
        b : a single number, the bias
    Returns:
        a single number, the "turnout score"
    """
    # TODO: calculate the final score using x, w, and b
    #       Hint: NumPy's @ operator does a dot product, e.g.  w @ x
    score = ____        # <-- replace ____
    return score

Run the check below. In the lecture we computed this score by hand and got **1.36** — your function should agree.

In [ ]:
# Self-check: this should reproduce the lecture's hand-computed score of 1.36.
score = forward(x_demo, w_demo, b_demo)
print(f"Turnout score for the demo voter: {score:.3f}")
assert np.isclose(score, 1.363, atol=0.01), "That doesn't match the lecture's 1.36."
print("Matches the lecture's hand-computed 1.36  [ok]")

A score of **1.36** is a confident "likely to vote".

> $y = \mathbf{w}\cdot\mathbf{x} + b$ is just **linear regression** with the symbols relabelled. A single linear neuron can only draw straight lines. We need to *stack* neurons and add a *non-linearity* (Section 3) to approximate more complex functions.

## Section 2 · The whole dataset at once (vectorising) + probabilities

Calling `forward` once per voter in a Python loop would be slow. The trick from the lecture is to stack all voters into the matrix $\mathbf{X}$ (one voter per **row**) and do **one matrix product**:

$$
\text{scores} = \mathbf{X}\,\mathbf{w} + b
$$

Shape check: $\mathbf{X}$ is $(n, 3)$ and $\mathbf{w}$ is $(3,)$, so $\mathbf{X}\,\mathbf{w}$ is $(n,)$ — one score per voter. (This is the row-vector convention we'll use all course.)

Let's pick weights for the whole population — roughly the linear part of the signal: older, richer, and past-voters turn out more.

In [ ]:
# Hand-picked weights for the WHOLE population (the linear part of the signal).
w_lin = np.array([1.6, 1.2, 2.0])
b_lin = -1.0

In [ ]:
# Vectorised forward pass over EVERY voter at once:
#   X has shape (n, 3);  w_lin has shape (3,);  X @ w_lin has shape (n,).
scores = X @ w_lin + b_lin
print("scores shape:", scores.shape)
print("first 5 scores:", np.round(scores[:5], 3))

### Using sigmoid to get probabilities

A raw score like `1.36` or `-0.4` isn't a probability. The **sigmoid** squashes any real number into $(0, 1)$ so we can read it as $P(\text{turnout})$:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

It's the first **activation function** from the lecture: $\sigma(0) = 0.5$, large positive $z \to 1$, large negative $z \to 0$.

**Your turn (TODO).** Implement `sigmoid`. Use `np.exp`.

In [ ]:
def sigmoid(z):
    """Squash any real number(s) z into the range (0, 1).

    Maths:  sigmoid(z) = 1 / (1 + e^(-z))

    Works element-wise on a NumPy array.
    """
    # TODO: return the probability corresponding to score z
    return ____

In [ ]:
# Self-check: sigmoid(0) is exactly 0.5; big inputs saturate near 0 and 1.
assert np.isclose(sigmoid(0.0), 0.5), "sigmoid(0) should be exactly 0.5"
assert sigmoid(20) > 0.99 and sigmoid(-20) < 0.01
probs = sigmoid(scores)
print("first 5 turnout probabilities:", np.round(probs[:5], 3))
print("sigmoid looks good  [ok]")

### From probabilities to a decision — the threshold

To make a hard yes/no prediction we apply a **threshold**: predict turnout (1) if the probability is at least 0.5, otherwise 0. Then we can measure **accuracy** — the fraction of voters we got right.

**Your turn (TODO).** Implement the threshold rule.

In [ ]:
def predict_from_probs(probs, threshold=0.5):
    """Turn probabilities into hard 0/1 predictions.

    A voter is predicted to turn out (1) if their probability is at least
    `threshold`, otherwise not (0).
    """
    # TODO: return an int array that is 1 where probs >= threshold, else 0.
    #       Hint: (probs >= threshold) gives True/False; .astype(int) -> 1/0.
    return ____

In [ ]:
# Turn probabilities into 0/1 predictions and measure accuracy.
preds = predict_from_probs(probs)
accuracy = (preds == y).mean()
print(f"Hand-picked perceptron accuracy: {accuracy:.1%}")
majority = max(y.mean(), 1 - y.mean())
print(f"(Always guessing the majority class would get {majority:.1%}.)")

Our hand-picked perceptron does a bit better than blindly guessing the majority class — but only a bit. We never *tuned* the weights to this data; we just guessed sensible ones. A *fitted* linear model would do better (we'll see one in Section 4).

## Section 3 · A two-layer network (forward pass only)

Now we can **stack** layers. A two-layer network adds a **hidden layer** between the inputs and the output, with a **non-linearity** in between. From Lecture 2:

$$
\mathbf{h} = \text{ReLU}\!\left(\mathbf{X}\,\mathbf{W}_1^{\mathsf T} + \mathbf{b}_1\right),
\qquad
\hat{\mathbf{y}} = \mathbf{h}\,\mathbf{w}_2 + b_2
$$

- $\mathbf{W}_1$ is the hidden layer's weights, shape `(hidden, inputs)` — the lecture's $\mathbf{W}\in\mathbb{R}^{h\times k}$, used as $\mathbf{X}\mathbf{W}^{\mathsf T}$.
- **ReLU**$(z) = \max(0, z)$ is the crucial non-linear step. Without it, two linear layers would collapse into one (the lecture's "depth alone buys you nothing").
- $\mathbf{w}_2, b_2$ are the output layer, which combines the hidden units into one score.

So the *entire* network is just a sequence of fairly simple calculations: a matmul, a non-linearity, another matmul.

> **Important — we can't TRAIN this yet.** We're going to set the weights at **random** and only run the forward pass. Computing the gradients that would let us *learn* good weights is **Thursday's Lab 4** (your own autograd); doing it in a framework is **Monday's Lab 5** (PyTorch). Today is about seeing the machinery.

In [ ]:
# Random starting weights for a 2-layer network (we are NOT training it yet).
rng = np.random.default_rng(0)
n_hidden = 8

# Layer 1 (hidden layer): width n_hidden; each unit sees all 3 inputs.
W1 = rng.normal(size=(n_hidden, 3))   # shape (hidden, inputs) -> lecture's W is (h x k)
b1 = np.zeros(n_hidden)               # one bias per hidden unit

# Layer 2 (output layer): combine the hidden units into one score.
w2 = rng.normal(size=n_hidden)        # shape (hidden,)
b2 = 0.0

print("W1:", W1.shape, "  b1:", b1.shape, "  w2:", w2.shape, "  b2:", b2)

**Your turn (TODO).** Implement the 2-layer forward pass — three lines, as written in the docstring. Use `np.maximum(0, ...)` for the ReLU.

In [ ]:
def forward2(X, W1, b1, w2, b2):
    """Forward pass of a 2-layer network (hidden ReLU layer + linear output).

    Following Lecture 2's convention (samples are rows):
        H_pre = X @ W1.T + b1      # (n, hidden) pre-activations
        H     = ReLU(H_pre)        # (n, hidden) apply the non-linearity
        out   = H @ w2 + b2        # (n,)        output scores

    ReLU(z) = max(0, z), element-wise -> use np.maximum(0, ...).
    """
    # TODO: implement the three lines described above.
    H_pre = ____
    H = ____
    out = ____
    return out

In [ ]:
# Run the 2-layer forward pass and check the output shape.
out = forward2(X, W1, b1, w2, b2)
print("output shape:", out.shape, "(one score per voter)")

rand_preds = predict_from_probs(sigmoid(out))
print(f"Accuracy of the UNTRAINED 2-layer net: {(rand_preds == y).mean():.1%}")
print("Near chance - and that's expected: the weights are RANDOM.")
print("Finding GOOD weights = training: Lab 4 (by hand) and Lab 5 (PyTorch).")

As promised, a network with **random** weights is no better than guessing. The architecture is correct; what's missing is *learning*. That's the cliffhanger the next two labs resolve.

## Section 4 · Baselines and the payoff

We can't train our NumPy net by hand yet — so let `scikit-learn` train two models for us, as a preview of where this is heading:

- **`LogisticRegression`** — a *trained* perceptron (Section 1/2, but with fitted weights).
- **`MLPClassifier`** — a *trained* two-layer net with a ReLU hidden layer (Section 3, but fitted).

We'll judge them on the held-out **test set** from Section 0.

In [ ]:
# Fit two TRAINED models on the training split.
#   LogisticRegression = a trained linear perceptron (Section 1/2, but fitted).
#   MLPClassifier      = a trained 2-layer net (Section 3, but fitted).
logreg = LogisticRegression().fit(X_train, y_train)
mlp = MLPClassifier(hidden_layer_sizes=(32,), activation="relu",
                    max_iter=3000, random_state=0).fit(X_train, y_train)

acc_logreg = accuracy_score(y_test, logreg.predict(X_test))
acc_mlp = accuracy_score(y_test, mlp.predict(X_test))
print(f"Logistic regression (trained perceptron):  {acc_logreg:.1%}")
print(f"MLP / 2-layer net   (trained 2-layer net): {acc_mlp:.1%}")

### Read this carefully

On raw test accuracy the two models are **basically tied**. Both have hit a *ceiling set by the noise* in the labels (turnout is only partly predictable from three features). 

To *see* the difference, we have to look at the **decision boundary** each model draws. Below, for each model we shade the predicted $P(\text{turnout})$ over the age × income plane, separately for voters who did and didn't vote in 2019. The **dashed black line** is the *true* rule that actually generated the data.

In [ ]:
def plot_boundary(ax, model, voted_value, title):
    """Shade P(turnout) over the age x income plane for a fixed voted2019 value."""
    g = np.linspace(0, 1, 200)
    aa, ii = np.meshgrid(g, g)
    grid = np.stack([aa.ravel(), ii.ravel(),
                     np.full(aa.size, voted_value)], axis=1).astype(np.float32)

    # The model's predicted probability of turnout, as a coloured surface.
    proba = model.predict_proba(grid)[:, 1].reshape(aa.shape)
    ax.contourf(aa, ii, proba, levels=np.linspace(0, 1, 21), cmap="RdBu", alpha=0.85)

    # The TRUE boundary that generated the data (dashed black line).
    ztrue = (1.6 * aa + 1.2 * ii + 2.0 * voted_value - 3.0 * aa * ii - 1.0)
    ptrue = 1 / (1 + np.exp(-ztrue))
    ax.contour(aa, ii, ptrue, levels=[0.5], colors="k",
               linestyles="--", linewidths=2)

    # The actual voters who have this voted2019 value, coloured by true label.
    mask = X[:, 2] == voted_value
    ax.scatter(X[mask, 0], X[mask, 1], c=y[mask], cmap="RdBu",
               s=8, edgecolors="k", linewidths=0.2)
    ax.set_xlabel("age01"); ax.set_ylabel("income01")
    ax.set_title(title)

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
plot_boundary(axes[0, 0], logreg, 0.0, "Logistic regression  ·  voted2019 = 0")
plot_boundary(axes[0, 1], logreg, 1.0, "Logistic regression  ·  voted2019 = 1")
plot_boundary(axes[1, 0], mlp,    0.0, "2-layer MLP  ·  voted2019 = 0")
plot_boundary(axes[1, 1], mlp,    1.0, "2-layer MLP  ·  voted2019 = 1")
fig.suptitle("Decision boundaries  (dashed = the TRUE rule that made the data)",
             fontsize=13)
plt.tight_layout(); plt.show()

### What to notice (the age × income interaction)

Look at the **`voted2019 = 0`** column (left):

- The true rule (dashed) is **curved**. Turnout is likely when *either* age *or* income is high — **but not both** (when both are high, the interaction term suppresses turnout). This is the lecture's **XOR-like** pattern.
- **Logistic regression** can only draw a **straight line**, so it cannot carve out that curved region — it essentially gives up and predicts "won't vote" for almost everyone in this group.
- The **MLP** bends its boundary to follow the true curve. *This* is what the hidden layer + ReLU buys you.

In the **`voted2019 = 1`** column (right) almost everyone turns out regardless of age/income, so both models agree — there's no curvature to capture. Past behaviour dominates.

Let's put numbers on it: how well does each model recover the *true* rule (ignoring label noise)?

In [ ]:
# Quantify the pictures: how well does each model recover the TRUE rule,
# ignoring label noise? Score against the noiseless boundary on a dense grid.
g = np.linspace(0, 1, 200)
aa, ii = np.meshgrid(g, g)
for voted_value in (0.0, 1.0):
    grid = np.stack([aa.ravel(), ii.ravel(),
                     np.full(aa.size, voted_value)], axis=1).astype(np.float32)
    ztrue = (1.6 * grid[:, 0] + 1.2 * grid[:, 1] + 2.0 * voted_value
             - 3.0 * grid[:, 0] * grid[:, 1] - 1.0)
    bayes = (1 / (1 + np.exp(-ztrue)) >= 0.5).astype(int)
    a_lr = (logreg.predict(grid) == bayes).mean()
    a_mlp = (mlp.predict(grid) == bayes).mean()
    print(f"voted2019={voted_value:.0f}:  logreg matches true rule {a_lr:.1%}"
          f"   |   MLP matches true rule {a_mlp:.1%}")

Among 2019 **non-voters**, the MLP matches the true rule noticeably better than logistic regression — it found the interaction. Among 2019 **voters**, they're identical. So the two-layer model wins *exactly where there's a non-linear interaction to capture*, and nowhere else.

> **Where this is going.** In **Lab 4** you'll implement this training yourself, with your own autograd engine. In **Lab 5** you'll do it in PyTorch — on this very dataset.

## Extension

Write a short paragraph (3–5 sentences) answering the question from the lecture brief:

> **Where does the two-layer model beat the perceptron? Where doesn't it?**

Use the evidence you just produced: the (tied) test accuracies, the decision-boundary plots, and the "matches the true rule" percentages. Mention the age × income interaction and the role of the ReLU non-linearity.

*(Double-click this cell to edit it, and type your answer below.)*

---

**Your answer:**

_…write here…_

## Recap, and a teaser for Lab 3

Today you:

- Built a perceptron as `forward(x, w, b) = w @ x + b`, and saw it *is* linear/logistic regression.
- Vectorised it over a whole dataset, added a **sigmoid** to get probabilities, and a **threshold** to get predictions.
- Stacked it into a **two-layer net** (matmul → **ReLU** → matmul) and confirmed: with random weights, it can't do anything useful.
- Saw the payoff — a *trained* two-layer net captures the age × income interaction that a linear model structurally cannot.

The one thing we couldn't do is the most important: **find good weights**. For that we need gradients, and an algorithm to compute them automatically.

**Lab 3 (tomorrow): we build the autograd engine** — the `Value` class and the computational graph — that will let us *train* these networks in Lab 4. See you there.

## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — forward**

In [ ]:
def forward(x, w, b):
    return w @ x + b

**Solution — sigmoid**

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

**Solution — predict_from_probs**

In [ ]:
def predict_from_probs(probs, threshold=0.5):
    return (probs >= threshold).astype(int)

**Solution — forward2**

In [ ]:
def forward2(X, W1, b1, w2, b2):
    H_pre = X @ W1.T + b1      # (n, hidden) pre-activations
    H = np.maximum(0, H_pre)   # (n, hidden) ReLU non-linearity
    out = H @ w2 + b2          # (n,)        output scores
    return out